# Aula 04: o simulador de carteiras em turnos, turnos 1 e 2

Módulo 08 AdmTech, Computação · Semana 6 · Sprint 3 · Prof. José Romualdo

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josercf/2026-2B-M08/blob/main/notebooks/aula04.ipynb)

Neste notebook você escreve à mão, sem IA, as duas primeiras funções do simulador: `retorno_do_dia` (turno 1) e `metricas` (turno 2). O colega ao lado revisa.

Cada turno segue a mesma sequência:

1. Ler a spec do turno.
2. Digitar o valor esperado, lido no portal de simulação para a sua carteira.
3. Rodar a conferência e ver **AINDA NÃO**, porque a função está vazia.
4. Escrever o corpo da função.
5. Rodar de novo até ver **OK**.

O que você entrega ao fim do bloco: os dois turnos em **OK** e o arquivo `simulador.py`, gravado pela última célula, que a sua dupla leva para o kit dos turnos 3 a 5.

Comece por **Ambiente de execução > Executar tudo**. O notebook roda inteiro com as respostas vazias.

In [ ]:
import inspect
import os

import numpy as np
import pandas as pd

# Os outros notebooks do módulo leem o snapshot do case em
# https://raw.githubusercontent.com/josercf/2026-2B-M08/main/dados/ .
# Este lê as cotações da aula, com preço de referência de 13/11/2026.
PRECOS_LOCAL = os.path.join("..", "exemplos", "aula04-simulador", "dados", "precos.csv")
PRECOS_BRUTO = ("https://raw.githubusercontent.com/josercf/2026-2B-M08/main/"
                "exemplos/aula04-simulador/dados/precos.csv")


def ler_precos():
    """Lê as cotações da aula: do disco quando o repositório está clonado, da URL bruta no Colab."""
    if os.path.exists(PRECOS_LOCAL):
        return pd.read_csv(PRECOS_LOCAL, index_col="data")
    if os.environ.get("M08_OFFLINE") == "1":
        raise FileNotFoundError("M08_OFFLINE=1 e %s não existe localmente" % PRECOS_LOCAL)
    return pd.read_csv(PRECOS_BRUTO, index_col="data")


def tabela_de_retornos(precos, ativos, pregoes):
    """Retornos diários dos últimos `pregoes` pregões, na janela comum dos ativos pedidos."""
    fora = [a for a in ativos if a not in precos.columns]
    if fora:
        raise ValueError("ativo não encontrado no arquivo da aula: %s" % ", ".join(fora))
    comum = precos[list(ativos)].dropna()
    tabela = comum.pct_change().iloc[1:]
    if len(tabela) < pregoes:
        raise ValueError("a janela comum tem %d retornos e foram pedidos %d" % (len(tabela), pregoes))
    return tabela.tail(pregoes)


precos = ler_precos()
print("%d ativos, pregões de %s a %s" % (precos.shape[1], precos.index[0], precos.index[-1]))

## 1. A sua carteira

Preencha `ATIVOS` com os tickers da carteira que você montou no portal (de 2 a 5). Mantenha `PREGOES = 9`, o mesmo período escolhido no portal, e os pesos iguais. Rode a célula.

In [ ]:
ATIVOS = ["PETR4", "VALE3"]                    # de 2 a 5 tickers, os mesmos do portal
PREGOES = 9                                    # período do portal: 9 pregões
PESOS = [1 / len(ATIVOS)] * len(ATIVOS)        # pesos iguais

print("Carteira:", ", ".join("%s %.1f%%" % (a, 100 * p) for a, p in zip(ATIVOS, PESOS)))

In [ ]:
#@title Conferência do professor (célula recolhida: rode, não precisa abrir)
# Conferência do professor. Calcula a referência por outro caminho (a série
# diária da carteira, com pandas) a partir de ATIVOS, PREGOES e PESOS, e compara
# três coisas: o valor que você digitou, o que a sua função devolve e a referência.
TOLERANCIA = 0.0005        # 0,05 ponto percentual, para o valor lido no portal
EXATO = 1e-9               # para a função, que faz a mesma conta da referência
PISTA_TURNO1 = ("Pista: o retorno da carteira no pregão é a soma de cada peso "
                "vezes o retorno do ativo naquele dia.")
PISTA_TURNO2 = ("Pista: calcule o retorno da carteira em cada pregão (uma linha da tabela "
                "por vez), depois tire a média e o desvio padrão dessa lista.")


def _pct(valor, casas=2):
    return ("%.*f%%" % (casas, 100 * float(valor))).replace(".", ",")


def _carteira():
    """Tabela de retornos, pesos e série diária da carteira descrita na célula 1."""
    if not 2 <= len(ATIVOS) <= 5:
        raise ValueError("ATIVOS precisa ter de 2 a 5 tickers e tem %d" % len(ATIVOS))
    pesos = np.asarray(PESOS, dtype=float)
    if len(pesos) != len(ATIVOS):
        raise ValueError("PESOS tem %d valores e ATIVOS tem %d" % (len(pesos), len(ATIVOS)))
    if abs(pesos.sum() - 1) > 1e-6 or (pesos < 0).any():
        raise ValueError("PESOS precisa ser não negativo e somar 1")
    tabela = tabela_de_retornos(precos, ATIVOS, PREGOES)
    serie = tabela.mul(pesos, axis=1).sum(axis=1)
    return tabela, pesos, serie


def _ler_esperado(valor):
    """Devolve as leituras possíveis do valor digitado, em fração. Lista vazia: em branco."""
    if valor is None or isinstance(valor, bool):
        return []
    porcentagem = False
    if isinstance(valor, str):
        texto = valor.strip().replace(" ", "").replace("\u2212", "-")
        if texto == "":
            return []
        porcentagem = "%" in texto
        texto = texto.replace("%", "")
        if "," in texto:
            texto = texto.replace(".", "").replace(",", ".")
        numero = float(texto)
    else:
        numero = float(valor)
    if numero != numero:
        raise ValueError("não é um número")
    if porcentagem:
        return [numero / 100]
    # Número com módulo maior que 1,5 é lido como porcentagem. A outra leitura
    # fica de reserva, para "1,23" (em %) e "-1.70" (em fração) também valerem.
    if abs(numero) > 1.5:
        return [numero / 100, numero]
    return [numero, numero / 100]


def _digitado(nome, esperado, referencia, casas, onde):
    """Parte 1: o valor digitado contra a referência. Devolve (ok, leitura ou None, linha)."""
    try:
        leituras = _ler_esperado(esperado)
    except (TypeError, ValueError):
        return False, None, ("[falta] %s digitado: não consegui ler %r como número. "
                             "Use um formato como 0.0123, 1.23, \"1,23%%\" ou \"1,23\"."
                             % (nome, esperado))
    if not leituras:
        return False, None, ("[falta] %s digitado: em branco. Leia no portal %s e troque o None "
                             "pelo valor, por exemplo \"1,23%%\"." % (nome, onde))
    for leitura in leituras:
        if abs(leitura - referencia) <= TOLERANCIA:
            return True, leitura, "[ok]    %s digitado (%s) confere com a referência." % (
                nome, _pct(leitura, casas))
    return False, leituras[0], (
        "[falta] %s digitado: veio %s e a referência para esta carteira é %s. "
        "Pista: confira se os ativos, os pesos e o período do portal são os mesmos da célula 1 "
        "e se você leu %s." % (nome, _pct(leituras[0], casas), _pct(referencia, casas), onde))


def _chamar(funcao, *argumentos):
    """Chama a função do aluno sem deixar erro escapar. Devolve (situação, valor ou texto)."""
    try:
        valor = funcao(*argumentos)
    except NotImplementedError:
        return "vazia", None
    except Exception as erro:  # o notebook nunca para por erro da função do aluno
        return "erro", "%s: %s" % (type(erro).__name__, erro)
    if valor is None:
        return "vazia", None
    return "valor", valor


def _numero(valor):
    try:
        numero = float(valor)
    except (TypeError, ValueError):
        return None
    return None if numero != numero else numero


def _funcao_turno1(funcao):
    """Parte 2 do turno 1: a função contra a referência, nos pregões da tabela."""
    tabela, pesos, serie = _carteira()
    for posicao in range(len(tabela)):
        linha = tabela.iloc[posicao].to_numpy()
        situacao, valor = _chamar(funcao, pesos, linha)
        dia = tabela.index[posicao]
        if situacao == "vazia":
            return False, None, ("[falta] Função: retorno_do_dia ainda não devolve nada. Escreva o "
                                 "corpo (uma linha começando com return). " + PISTA_TURNO1)
        if situacao == "erro":
            return False, None, ("[falta] Função: retorno_do_dia parou com o erro %s. %s"
                                 % (valor, PISTA_TURNO1))
        numero = _numero(valor)
        if numero is None:
            return False, None, ("[falta] Função: retorno_do_dia devolveu %r, e o esperado é um "
                                 "número só. %s" % (valor, PISTA_TURNO1))
        referencia = float(serie.iloc[posicao])
        if posicao == 0:
            primeiro = numero
        if abs(numero - referencia) > EXATO:
            if posicao > 0:
                motivo = ("A função acerta o primeiro pregão e erra o de %s: ela precisa calcular "
                          "a partir dos dois argumentos, sem número fixo." % dia)
            elif abs(numero - linha.sum()) <= EXATO:
                motivo = "O que veio é a soma dos retornos dos ativos, sem os pesos."
            elif abs(numero - linha.mean()) <= EXATO:
                motivo = "O que veio é a média simples dos retornos, sem os pesos."
            else:
                motivo = ""
            return False, primeiro if posicao else numero, (
                "[falta] Função: no pregão de %s retorno_do_dia devolveu %s e a referência é %s. "
                "%s %s" % (dia, _pct(numero, 4), _pct(referencia, 4), motivo, PISTA_TURNO1))
    return True, primeiro, ("[ok]    Função: retorno_do_dia confere com a referência nos %d pregões."
                            % len(tabela))


def _conferir_turno1(esperado, funcao):
    tabela, pesos, serie = _carteira()
    dia = tabela.index[0]
    referencia = float(serie.iloc[0])
    onde = "o retorno da carteira no primeiro pregão (%s)" % dia
    ok_digitado, lido, linha_digitado = _digitado("Valor", esperado, referencia, 4, onde)
    ok_funcao, devolvido, linha_funcao = _funcao_turno1(funcao)
    linhas = [linha_digitado, linha_funcao]
    if lido is not None and devolvido is not None:
        if abs(devolvido - lido) <= TOLERANCIA:
            linhas.append("[ok]    Função e valor digitado: %s e %s, dentro de 0,05 ponto percentual."
                          % (_pct(devolvido, 4), _pct(lido, 4)))
        else:
            linhas.append("[falta] Função e valor digitado: a função devolveu %s e você digitou %s."
                          % (_pct(devolvido, 4), _pct(lido, 4)))
    else:
        linhas.append("[falta] Função e valor digitado: só dá para comparar quando os dois existem.")
    passou = ok_digitado and ok_funcao
    print("TURNO 1: %s" % ("OK" if passou else "AINDA NÃO"))
    for linha in linhas:
        print("  " + linha)
    return passou


def conferir_turno1(esperado, funcao):
    """Confere o turno 1 e responde OK ou AINDA NÃO, sem parar o notebook."""
    try:
        return _conferir_turno1(esperado, funcao)
    except Exception as erro:
        print("TURNO 1: AINDA NÃO")
        print("  [falta] A conferência não rodou: %s. Confira ATIVOS, PREGOES e PESOS na "
              "célula 1 e rode as células de cima de novo." % erro)
        return False


def _funcao_turno2(funcao):
    """Parte 2 do turno 2: a função contra a referência. Devolve (ok, par ou None, linhas)."""
    tabela, pesos, serie = _carteira()
    ref_retorno = float(serie.mean() * 252)
    ref_risco = float(serie.std(ddof=1) * np.sqrt(252))
    situacao, valor = _chamar(funcao, pesos, tabela)
    if situacao == "vazia":
        return False, None, ["[falta] Função: metricas ainda não devolve nada. Escreva o corpo "
                             "(até três linhas, a última começando com return). " + PISTA_TURNO2]
    if situacao == "erro":
        return False, None, ["[falta] Função: metricas parou com o erro %s. %s" % (valor, PISTA_TURNO2)]
    try:
        retorno, risco = valor
        retorno, risco = _numero(retorno), _numero(risco)
    except (TypeError, ValueError):
        retorno = risco = None
    if retorno is None or risco is None:
        return False, None, ["[falta] Função: metricas devolveu %r, e o esperado são dois números, "
                             "nesta ordem: return retorno_anual, risco_anual." % (valor,)]
    linhas = []
    if abs(retorno - ref_retorno) <= EXATO:
        linhas.append("[ok]    Função: o retorno anual (%s) confere com a referência." % _pct(retorno))
    else:
        if abs(retorno - serie.mean()) <= EXATO:
            motivo = "O que veio é a média diária: falta multiplicar por 252."
        elif abs(retorno - serie.sum()) <= EXATO:
            motivo = "O que veio é a soma dos retornos diários: a definição pede a média vezes 252."
        else:
            motivo = "A definição é a média dos retornos diários da carteira vezes 252."
        linhas.append("[falta] Função: o retorno anual veio %s e a referência é %s. %s"
                      % (_pct(retorno), _pct(ref_retorno), motivo))
    if abs(risco - ref_risco) <= EXATO:
        linhas.append("[ok]    Função: o risco anual (%s) confere com a referência." % _pct(risco))
    else:
        if abs(risco - serie.std(ddof=0) * np.sqrt(252)) <= EXATO:
            motivo = ("O que veio é o desvio padrão populacional. A definição pede o amostral: "
                      "passe ddof=1 ao calcular o desvio.")
        elif abs(risco - serie.std(ddof=1)) <= EXATO:
            motivo = "O que veio é o desvio padrão diário: falta multiplicar pela raiz de 252."
        elif abs(risco - serie.std(ddof=1) * 252) <= EXATO:
            motivo = "O risco anualiza pela raiz de 252, e o que veio foi multiplicado por 252."
        else:
            motivo = ("A definição é o desvio padrão amostral (ddof=1) dos retornos diários da "
                      "carteira vezes a raiz de 252.")
        linhas.append("[falta] Função: o risco anual veio %s e a referência é %s. %s"
                      % (_pct(risco), _pct(ref_risco), motivo))
    ok = all(linha.startswith("[ok]") for linha in linhas)
    if not ok:
        linhas.append("        " + PISTA_TURNO2)
    return ok, (retorno, risco), linhas


def _conferir_turno2(esperado_retorno, esperado_risco, funcao):
    tabela, pesos, serie = _carteira()
    ref_retorno = float(serie.mean() * 252)
    ref_risco = float(serie.std(ddof=1) * np.sqrt(252))
    ok_ret, lido_ret, linha_ret = _digitado(
        "Retorno anual", esperado_retorno, ref_retorno, 2, "o retorno anual da carteira")
    ok_risco, lido_risco, linha_risco = _digitado(
        "Risco anual", esperado_risco, ref_risco, 2, "o risco anual da carteira")
    ok_funcao, devolvido, linhas_funcao = _funcao_turno2(funcao)
    linhas = [linha_ret, linha_risco] + linhas_funcao
    if devolvido is not None and lido_ret is not None and lido_risco is not None:
        if abs(devolvido[0] - lido_ret) <= TOLERANCIA and abs(devolvido[1] - lido_risco) <= TOLERANCIA:
            linhas.append("[ok]    Função e valores digitados: dentro de 0,05 ponto percentual.")
        else:
            linhas.append("[falta] Função e valores digitados: a função devolveu %s e %s, e você "
                          "digitou %s e %s." % (_pct(devolvido[0]), _pct(devolvido[1]),
                                               _pct(lido_ret), _pct(lido_risco)))
    else:
        linhas.append("[falta] Função e valores digitados: só dá para comparar quando os três existem.")
    passou = ok_ret and ok_risco and ok_funcao
    print("TURNO 2: %s" % ("OK" if passou else "AINDA NÃO"))
    for linha in linhas:
        print("  " + linha)
    return passou


def conferir_turno2(esperado_retorno, esperado_risco, funcao):
    """Confere o turno 2 e responde OK ou AINDA NÃO, sem parar o notebook."""
    try:
        return _conferir_turno2(esperado_retorno, esperado_risco, funcao)
    except Exception as erro:
        print("TURNO 2: AINDA NÃO")
        print("  [falta] A conferência não rodou: %s. Confira ATIVOS, PREGOES e PESOS na "
              "célula 1 e rode as células de cima de novo." % erro)
        return False


def _funcoes_prontas(funcao_turno1, funcao_turno2):
    """Verdadeiro quando as duas funções conferem com a referência."""
    try:
        return _funcao_turno1(funcao_turno1)[0] and _funcao_turno2(funcao_turno2)[0]
    except Exception:
        return False


print("Conferência carregada: conferir_turno1 e conferir_turno2.")

## 2. Retornos diários dos 9 pregões

A tabela abaixo é a mesma que o portal mostra para a sua carteira: uma linha por pregão, uma coluna por ativo, em porcentagem. A variável `retornos` guarda os mesmos números em fração (1,23% é `0.0123`).

In [ ]:
retornos = tabela_de_retornos(precos, ATIVOS, PREGOES)
(retornos * 100).round(2)

## 3. Turno 1: `retorno_do_dia`

**Spec.** `retorno_do_dia(pesos, retornos_do_dia)` recebe os pesos da carteira e os retornos dos ativos em um pregão e devolve o retorno da carteira nesse pregão.

Exemplo com números redondos, para conferir de cabeça: pesos de 50% e 50%, retornos de 2% e -1% no pregão, retorno da carteira de 0,5%.

A célula abaixo traz a assinatura e a docstring. O corpo é uma linha.

In [ ]:
def retorno_do_dia(pesos, retornos_do_dia):
    """Retorno da carteira em um pregão.

    pesos: lista com o peso de cada ativo, na ordem de ATIVOS (soma 1).
    retornos_do_dia: lista com o retorno de cada ativo no pregão, na mesma ordem.
    Devolve um número: o retorno da carteira no pregão, em fração.
    """
    # Escreva o corpo aqui: uma linha, começando com return.

**Teste do turno 1.** No portal, leia o retorno da sua carteira no **primeiro** dos 9 pregões e digite em `ESPERADO_TURNO1`, no lugar de `None`. Valem `0.0123`, `1.23`, `"1,23%"` e `"1,23"`.

Rode a célula antes de escrever a função: a resposta é **AINDA NÃO**. Depois de escrever o corpo, rode a célula da função e esta de novo.

In [ ]:
ESPERADO_TURNO1 = None    # o retorno da carteira no primeiro pregão, lido no portal

conferir_turno1(ESPERADO_TURNO1, retorno_do_dia);

## 4. Turno 2: `metricas`

**Spec.** `metricas(pesos, retornos)` recebe os pesos e a tabela de retornos de N pregões e devolve dois números: o retorno anual e o risco anual da carteira.

Ferramentas que o corpo usa, em até três linhas:

| Para | Use |
|---|---|
| percorrer os pregões | `retornos.values`, que entrega uma linha da tabela por vez |
| aplicar uma função a cada item | `[funcao(item) for item in lista]` |
| média de uma lista | `np.mean(lista)` |
| desvio padrão amostral | `np.std(lista, ddof=1)` |
| raiz quadrada | `np.sqrt(252)` |

A função do turno 1 já calcula o retorno da carteira em um pregão.

In [ ]:
def metricas(pesos, retornos):
    """Retorno anual e risco anual da carteira em N pregões.

    pesos: lista com o peso de cada ativo, na ordem das colunas de `retornos`.
    retornos: tabela com uma linha por pregão e uma coluna por ativo, em fração.

    Definições:
    - retorno da carteira no pregão: soma dos pesos vezes os retornos do pregão;
    - retorno anual: média dos retornos diários da carteira vezes 252;
    - risco anual: desvio padrão amostral (ddof=1) dos retornos diários da
      carteira vezes a raiz de 252.

    Devolve dois números, nesta ordem: (retorno_anual, risco_anual), em fração.
    """
    # Escreva o corpo aqui: até três linhas, a última começando com return.

**Teste do turno 2.** No portal, leia o retorno anual e o risco anual da sua carteira nos 9 pregões e digite nos dois campos. Rode antes de escrever a função para ver **AINDA NÃO**, e de novo depois.

In [ ]:
ESPERADO_RETORNO_ANUAL = None    # lido no portal, por exemplo "-12,3%"
ESPERADO_RISCO_ANUAL = None      # lido no portal, por exemplo "24,8%"

conferir_turno2(ESPERADO_RETORNO_ANUAL, ESPERADO_RISCO_ANUAL, metricas);

## 5. Levar as duas funções para o kit

Com os dois turnos em **OK**, a célula abaixo grava `simulador.py` com as duas funções. No Colab, o arquivo é baixado para o seu computador (ele também aparece no painel de arquivos, à esquerda). A sua dupla cola as duas funções no `simulador.py` do kit, no lugar das versões vazias, seguindo o `README.md` do kit.

In [ ]:
if _funcoes_prontas(retorno_do_dia, metricas):
    with open("simulador.py", "w", encoding="utf-8") as arquivo:
        arquivo.write('"""Turnos 1 e 2 do simulador de carteiras, escritos à mão no Colab."""\n')
        arquivo.write("import numpy as np\nimport pandas as pd\n\n\n")
        arquivo.write(inspect.getsource(retorno_do_dia).rstrip() + "\n\n\n")
        arquivo.write(inspect.getsource(metricas).rstrip() + "\n")
    print("simulador.py gravado com retorno_do_dia e metricas.")
    try:
        from google.colab import files
        files.download("simulador.py")
    except ImportError:
        print("Fora do Colab: o arquivo está na pasta deste notebook.")
else:
    print("AINDA NÃO: simulador.py só é gravado quando as duas funções conferem com a "
          "referência. Volte aos turnos 1 e 2 e leia a mensagem de cada conferência.")